# Trayectorias por paradigma (Semana 3–4)

Lee JSON de `results/sweep/` o `results/smoke/`, arma series por paradigma y checkpoint, ajusta polinomio grado 5 (log-tokens) y clasifica topología (monótona / U / U invertida / oscilatoria). Ver `docs/04_experimental_design.md`.

Tras corridas, generar tabla larga con `python -m ontogenia aggregate --output-parquet ../results/aggregated_metrics.parquet` y cargar ese Parquet aquí como alternativa a parsear muchos JSON.

In [1]:
from pathlib import Path

import pandas as pd

from ontogenia.topology import classify_all_tasks

ROOT = Path("..")
PARQUET = ROOT / "results" / "aggregated_metrics.parquet"
PARQUET.exists()

True

In [2]:
if not PARQUET.exists():
    raise FileNotFoundError(
        "Falta aggregated_metrics.parquet. Ejecutar: "
        "python -m ontogenia aggregate --output-parquet results/aggregated_metrics.parquet"
    )

frame = pd.read_parquet(PARQUET)
# BLiMP tasks y métrica de accuracy principal del harness
traj = frame.dropna(subset=["training_step", "acc,none"]).copy()
traj = traj[traj["task"].astype(str).str.startswith("blimp")]

summary = classify_all_tasks(traj, metric_col="acc,none")
summary.sort_values(["model_size", "shape", "task"]).head(20)

,model_size,task,n_points,shape,sign_changes,min_step,max_step,depth,peak_height
0,14m,blimp,24,oscillatory,4,0.000000,99884.422111,0.142877,0.142877
1,14m,blimp_adjunct_island,24,oscillatory,2,0.000000,30180.904523,0.181468,0.181468
2,14m,blimp_anaphor_gender_agreement,24,oscillatory,4,97728.643216,0.000000,0.171986,0.000000
3,14m,blimp_anaphor_number_agreement,27,oscillatory,4,0.000000,20839.195980,0.390601,0.390601
4,14m,blimp_animate_subject_passive,24,oscillatory,4,0.000000,20120.603015,0.223405,0.223405
5,14m,blimp_animate_subject_trans,24,oscillatory,4,61798.994975,7904.522613,0.092683,0.011517
6,14m,blimp_causative,24,oscillatory,4,0.000000,24432.160804,0.257214,0.257214
7,14m,blimp_complex_NP_island,24,oscillatory,4,99165.829146,0.000000,0.149245,0.000000
8,14m,blimp_coordinate_structure_constraint_complex_...,24,oscillatory,4,20120.603015,0.000000,0.404228,0.000000
9,14m,blimp_coordinate_structure_constraint_object_e...,24,oscillatory,3,48864.321608,104195.979899,0.135389,0.025319


In [3]:
# Opcional: guardar clasificación topológica para paper/figuras
out = ROOT / "results" / "topology_summary.parquet"
summary.to_parquet(out, index=False)
out

PosixPath('../results/topology_summary.parquet')